# Day 2. 생성형AI를 활용한 데이터분석
- 가상 엑셀 파일 생성 후 병합

In [1]:
"""
<prompt>
  <task>
    주피터 노트북으로 작업중이다.
    새로운 폴더를 생성해서 가상의 엑셀 파일을 100개 가상 데이터로 만들어주는 코드를 작성하라.
  </task>
  <constraints>
    - 라이브러리가 필요한 경우 설치 코드를 작성하되 주석 처리하라.
  </constraints>
</prompt>
"""

# !pip install openpyxl faker pandas

import os
import random
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker

fake = Faker('ko_KR')
Faker.seed(42)
random.seed(42)

FOLDER = "excel_100_files"
os.makedirs(FOLDER, exist_ok=True)

departments = ["영업팀", "마케팅팀", "개발팀", "인사팀", "재무팀", "운영팀"]
positions = ["사원", "대리", "과장", "차장", "부장"]
statuses = ["재직", "휴직", "퇴직"]
regions = ["서울", "부산", "대구", "인천", "광주", "대전", "수원"]


def make_excel(file_idx):
    """월별 직원 현황 파일 1개 생성 (시트 3개)"""
    wb = Workbook()

    # ── 시트1: 직원명단 ──────────────────────────────
    ws1 = wb.active
    ws1.title = "직원명단"
    headers1 = ["사번", "이름", "부서", "직급", "입사일", "기본급(만원)", "성과점수", "재직상태", "근무지"]
    ws1.append(headers1)

    n_rows = random.randint(10, 30)
    base = datetime(2018, 1, 1)
    for i in range(1, n_rows + 1):
        ws1.append([
            f"EMP{file_idx:03d}-{i:03d}",
            fake.name(),
            random.choice(departments),
            random.choice(positions),
            (base + timedelta(days=random.randint(0, 2000))).strftime("%Y-%m-%d"),
            random.randint(2800, 9000),
            round(random.uniform(60, 100), 1),
            random.choices(statuses, weights=[80, 10, 10])[0],
            random.choice(regions),
        ])

    # ── 시트2: 급여내역 ──────────────────────────────
    ws2 = wb.create_sheet("급여내역")
    headers2 = ["사번", "이름", "기본급", "성과급", "식대", "교통비", "공제액", "실수령액"]
    ws2.append(headers2)
    for row in ws1.iter_rows(min_row=2, values_only=True):
        base_sal = row[5]
        perf = round(base_sal * random.uniform(0.05, 0.2))
        meal = 100
        trans = 50
        deduct = round((base_sal + perf) * 0.09)
        ws2.append([row[0], row[1], base_sal, perf, meal, trans, deduct,
                    base_sal + perf + meal + trans - deduct])

    # ── 시트3: 부서요약 ──────────────────────────────
    ws3 = wb.create_sheet("부서요약")
    ws3.append(["부서", "인원", "평균기본급", "평균성과점수"])
    df_tmp = pd.DataFrame(ws1.iter_rows(min_row=2, values_only=True), columns=headers1)
    summary = df_tmp.groupby("부서").agg(
        인원=("사번", "count"),
        평균기본급=("기본급(만원)", "mean"),
        평균성과점수=("성과점수", "mean")
    ).reset_index()
    for _, r in summary.iterrows():
        ws3.append([r["부서"], int(r["인원"]),
                    round(r["평균기본급"], 0), round(r["평균성과점수"], 1)])

    # ── 헤더 스타일 공통 적용 ────────────────────────
    fill = PatternFill("solid", start_color="2F5496")
    font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    align = Alignment(horizontal="center", vertical="center")
    for ws in [ws1, ws2, ws3]:
        for cell in ws[1]:
            cell.font, cell.fill, cell.alignment = font, fill, align
        ws.row_dimensions[1].height = 20

    fname = os.path.join(FOLDER, f"monthly_report_{file_idx:03d}.xlsx")
    wb.save(fname)
    return fname


# 100개 생성
for idx in range(1, 101):
    make_excel(idx)
    if idx % 10 == 0:
        print(f"  {idx}/100 완료...")

print(f"\n총 100개 파일 생성 → ./{FOLDER}/")

  10/100 완료...
  20/100 완료...
  30/100 완료...
  40/100 완료...
  50/100 완료...
  60/100 완료...
  70/100 완료...
  80/100 완료...
  90/100 완료...
  100/100 완료...

총 100개 파일 생성 → ./excel_100_files/


In [2]:
"""
<prompt>
  <task>
    100개의 엑셀 파일들이 어떻게 구성되어있는지 한 번에 확인할 수 있는 코드를 작성하라.
  </task>
</prompt>
"""

import glob

summary_rows = []

for fpath in sorted(glob.glob(os.path.join(FOLDER, "*.xlsx"))):
    fname = os.path.basename(fpath)
    xf = pd.ExcelFile(fpath)
    sheets = xf.sheet_names

    row = {"파일명": fname, "시트 수": len(sheets), "시트 목록": " / ".join(sheets)}

    for sheet in sheets:
        df = pd.read_excel(fpath, sheet_name=sheet)
        row[f"[{sheet}] 행수"] = len(df)
        row[f"[{sheet}] 열수"] = len(df.columns)

    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

# 출력 옵션 — 모든 열·행 표시
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 110)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)

print(f"=== 엑셀 파일 구성 요약 ({len(df_summary)}개) ===\n")
df_summary

# 파일별 직원명단 행 수 분포
col = "[직원명단] 행수"
print("▶ 직원명단 행 수 분포")
print(df_summary[col].describe().to_string())
print(f"\n  최소 {int(df_summary[col].min())}행  /  최대 {int(df_summary[col].max())}행  /  평균 {df_summary[col].mean():.1f}행")
print(f"\n▶ 전체 시트 종류: {df_summary['시트 목록'].unique()}")
print(f"▶ 총 데이터 행 수 합계: {df_summary[col].sum():,}행")

=== 엑셀 파일 구성 요약 (100개) ===

▶ 직원명단 행 수 분포
count    100.000000
mean      20.140000
std        6.164447
min       10.000000
25%       14.750000
50%       20.000000
75%       25.000000
max       30.000000

  최소 10행  /  최대 30행  /  평균 20.1행

▶ 전체 시트 종류: ['직원명단 / 급여내역 / 부서요약']
▶ 총 데이터 행 수 합계: 2,014행


In [3]:
"""
<prompt>
  <task>
    같은 컬럼들이 있으면 그걸 이어서 세로로 병합하려고 한다.
    가능한지 확인 후 가능하다면 1개의 파일로 병합하는 코드를 작성하라.
  </task>
</prompt>
"""

import glob
import pandas as pd
import os

FOLDER = "excel_100_files"

# 시트별로 각 파일의 컬럼 목록 수집
sheet_col_map = {}  # { sheet_name: { frozenset(columns): [파일명, ...] } }

for fpath in sorted(glob.glob(os.path.join(FOLDER, "*.xlsx"))):
    fname = os.path.basename(fpath)
    xf = pd.ExcelFile(fpath)
    for sheet in xf.sheet_names:
        df = pd.read_excel(fpath, sheet_name=sheet)
        cols = tuple(df.columns.tolist())
        if sheet not in sheet_col_map:
            sheet_col_map[sheet] = {}
        sheet_col_map[sheet].setdefault(cols, []).append(fname)

# 결과 출력
print("=" * 60)
for sheet, col_groups in sheet_col_map.items():
    print(f"\n📋 시트: [{sheet}]")
    if len(col_groups) == 1:
        cols = list(col_groups.keys())[0]
        count = len(list(col_groups.values())[0])
        print(f"  ✅ 100개 파일 모두 동일한 컬럼 구성 ({count}개 파일)")
        print(f"  컬럼 ({len(cols)}개): {list(cols)}")
    else:
        print(f"  ⚠️  컬럼 구성이 {len(col_groups)}가지로 다름")
        for cols, files in col_groups.items():
            print(f"  - {list(cols)} → {len(files)}개 파일")
print("=" * 60)

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

OUTPUT_PATH = "output/merged_all.xlsx"

wb_out = Workbook()
wb_out.remove(wb_out.active)  # 기본 빈 시트 제거

# 시트별로 병합
all_files = sorted(glob.glob(os.path.join(FOLDER, "*.xlsx")))

for sheet_name in sheet_col_map.keys():
    frames = []
    for fpath in all_files:
        fname = os.path.basename(fpath)
        xf = pd.ExcelFile(fpath)
        if sheet_name in xf.sheet_names:
            df = pd.read_excel(fpath, sheet_name=sheet_name)
            df.insert(0, "출처파일", fname)   # 어느 파일 출신인지 추적
            frames.append(df)

    df_merged = pd.concat(frames, ignore_index=True)

    ws = wb_out.create_sheet(title=sheet_name)

    # 헤더 스타일
    header_font  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    header_fill  = PatternFill("solid", start_color="2F5496")
    header_align = Alignment(horizontal="center", vertical="center")
    even_fill    = PatternFill("solid", start_color="DCE6F1")
    center_align = Alignment(horizontal="center", vertical="center")

    for r_idx, row in enumerate(dataframe_to_rows(df_merged, index=False, header=True)):
        ws.append(row)
        if r_idx == 0:  # 헤더 행
            for cell in ws[1]:
                cell.font      = header_font
                cell.fill      = header_fill
                cell.alignment = header_align
        else:           # 데이터 행
            for cell in ws[r_idx + 1]:
                cell.font      = Font(name="Arial", size=10)
                cell.alignment = center_align
                if (r_idx + 1) % 2 == 0:
                    cell.fill = even_fill

    # 열 너비 자동 조정
    for col in ws.columns:
        max_len = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 40)

    ws.row_dimensions[1].height = 20
    print(f"  [{sheet_name}] 병합 완료: {len(df_merged):,}행 × {len(df_merged.columns)}열")

wb_out.save(OUTPUT_PATH)
print(f"\n✅ 저장 완료: {OUTPUT_PATH}")

print("=" * 55)
print(f"{'시트명':<12} {'병합 행수':>10} {'열수':>6} {'출처파일 수':>10}")
print("-" * 55)

xf_out = pd.ExcelFile(OUTPUT_PATH)
for sheet in xf_out.sheet_names:
    df = pd.read_excel(OUTPUT_PATH, sheet_name=sheet)
    n_files = df["출처파일"].nunique()
    print(f"{sheet:<12} {len(df):>10,} {len(df.columns):>6} {n_files:>10}")

print("=" * 55)
print(f"\n원본 파일: {len(all_files)}개  →  병합 파일: {OUTPUT_PATH} (시트 {len(xf_out.sheet_names)}개)")


📋 시트: [직원명단]
  ✅ 100개 파일 모두 동일한 컬럼 구성 (100개 파일)
  컬럼 (9개): ['사번', '이름', '부서', '직급', '입사일', '기본급(만원)', '성과점수', '재직상태', '근무지']

📋 시트: [급여내역]
  ✅ 100개 파일 모두 동일한 컬럼 구성 (100개 파일)
  컬럼 (8개): ['사번', '이름', '기본급', '성과급', '식대', '교통비', '공제액', '실수령액']

📋 시트: [부서요약]
  ✅ 100개 파일 모두 동일한 컬럼 구성 (100개 파일)
  컬럼 (4개): ['부서', '인원', '평균기본급', '평균성과점수']
  [직원명단] 병합 완료: 2,014행 × 10열
  [급여내역] 병합 완료: 2,014행 × 9열
  [부서요약] 병합 완료: 577행 × 5열

✅ 저장 완료: merged_all.xlsx
시트명               병합 행수     열수     출처파일 수
-------------------------------------------------------
직원명단              2,014     10        100
급여내역              2,014      9        100
부서요약                577      5        100

원본 파일: 100개  →  병합 파일: merged_all.xlsx (시트 3개)
